# Evaluation of the Surfeit class

This notebook demonstrates surfeit scores and analysis of model descriptions.

Fitted models use the library's canonical serializers. Explicit description strings
can also be evaluated directly. Analysis reports expose the code lengths used in
surfeit and the limiting reference, without interpreting compression as proof of overfitting.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams

from sklearn.datasets import make_blobs
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier, export_text

from mnplib.inaccuracy import Inaccuracy
from mnplib.surfeit import Surfeit, surfeit_string


In [ ]:
rcParams["figure.figsize"] = 10, 5
rng = np.random.default_rng(42)

## A Trained Model
The model method obtains the canonical description through the serializer layer.
`pformat()` formats the numerical report and canonical model string for inspection.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from mnplib import Surfeit
from mnplib.surfeit import surfeit_model
from pprint import pformat

model_X, model_y = make_regression(n_samples=100, n_features=2, random_state=42)
fitted_model = LinearRegression().fit(model_X, model_y)
metric = Surfeit(n_bins=3).fit(model_X, model_y)
value = metric.surfeit_model(fitted_model)
description = metric.model_analysis(fitted_model)
assert np.isclose(value, metric.surfeit_string(description["model_string"]))
assert np.isclose(value, surfeit_model(fitted_model, X=model_X, y=model_y, n_bins=3))
print(pformat(description))

## Description Analysis
All code-length fields in analysis reports are in bits. The reference length is the
minimum of the target and overhead-corrected compressed code lengths. `reference_source`
identifies the limiting quantity. `compression_ratio` uses raw compressed and model
sizes before correction and may exceed one. `description_lengths()` is a separate
target-independent utility that returns byte counts.

In [ ]:
from mnplib.surfeit import description_analysis, model_analysis

text = description["model_string"]
analysis = metric.description_analysis(text)
assert analysis == description_analysis(text, y=model_y, n_bins=3)
assert description == model_analysis(fitted_model, X=model_X, y=model_y, n_bins=3)
byte_lengths = metric.description_lengths(text)
assert analysis["model_code_length_bits"] == 8 * byte_lengths["model_length"]
assert np.isclose(value, 1 - analysis["reference_code_length_bits"] / analysis["model_code_length_bits"])
print(pformat(analysis))

## 1. Basic string-based usage

The simplest use case is to fit the target representation and then evaluate a model description string.


In [ ]:
y = np.array([0, 0, 1, 1, 0, 1])

model_description = """def model(x):
    return int(x > 0)
"""

sft = Surfeit(y_type="categorical")
sft.fit_y(y)

sft.surfeit_string(model_description)

The same computation can be performed with the functional interface.


In [ ]:
surfeit_string(model_description, y_type='categorical', y=y)

## 2. Surfeit and redundant descriptions

Surfeit is sensitive to redundancy in the model description. The following two descriptions express the same simple rule, but the second one contains repeated, unnecessary structure.


In [ ]:
compact_rule = """def model(x):
    return int(x > 0)
"""

redundant_rule = """def model(x):
    # Repeated equivalent tests.
    if x > 0:
        result = 1
    else:
        result = 0

    if x > 0:
        result_again = 1
    else:
        result_again = 0

    if x > 0:
        result_third_time = 1
    else:
        result_third_time = 0

    return result
"""

rows = []
for name, description in [
    ("Compact rule", compact_rule),
    ("Redundant rule", redundant_rule),
]:
    rows.append(
        {
            "description": name,
            "length": len(description.encode("utf-8")),
            "surfeit": sft.surfeit_string(description),
        }
    )

pd.DataFrame(rows)

The redundant description is longer and more compressible. This is precisely the kind of unnecessary descriptive structure that surfeit is intended to expose.


## 3. Surfeit and polynomial overfitting

In this example, we approximate a noisy sinusoidal curve with polynomial models of increasing degree.

This example defines polynomial descriptions explicitly for the string-based API. Fitted estimators can also be evaluated through `surfeit_model()`, which uses the library's canonical serializers.


In [ ]:
def linear_model_to_string(model, *, name="model") -> str:
    """Serialize a fitted linear model as a compact Python-like expression."""
    coefs = np.ravel(model.coef_)
    terms = [f"{coef:.6g} * x{i}" for i, coef in enumerate(coefs)]
    expression = " + ".join(terms) if terms else "0"
    expression = f"{model.intercept_:.6g} + " + expression

    return f"""def {name}(x):
    return {expression}
"""


def polynomial_model_to_string(pipeline, degree: int) -> str:
    """Serialize a fitted polynomial regression pipeline."""
    model = pipeline.named_steps["linearregression"]
    description = linear_model_to_string(model, name=f"polynomial_degree_{degree}")
    return "# PolynomialFeatures are assumed to be applied before this function.\n" + description


In [ ]:
n_samples = 900
degrees = np.arange(1, 16)

X = np.sort(rng.random(n_samples) * 3)
noise = 0.08 * rng.normal(size=n_samples)
y = np.cos(1.5 * np.pi * X) + noise

plt.scatter(X, y, s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy sinusoidal target")
plt.show()

In [ ]:
inacc = Inaccuracy(y_type="numeric", n_bins="auto")
sft = Surfeit(y_type="numeric", n_bins="auto")

inaccuracies = []
surfeits = []
description_lengths = []

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    model.fit(X[:, np.newaxis], y)

    predictions = model.predict(X[:, np.newaxis])
    description = polynomial_model_to_string(model, degree=degree)

    inacc.fit_y(y)
    sft.fit_y(y)

    inaccuracies.append(inacc.inaccuracy_predictions(predictions))
    surfeits.append(sft.surfeit_string(description))
    description_lengths.append(len(description.encode("utf-8")))

In [ ]:
plt.plot(degrees, inaccuracies, marker="o", label="Inaccuracy")
plt.plot(degrees, surfeits, marker="o", label="Surfeit")
plt.xlabel("Polynomial degree")
plt.ylabel("Metric value")
plt.title("Inaccuracy decreases while surfeit may increase")
plt.legend(loc="best")
plt.show()

In [ ]:
pd.DataFrame(
    {
        "degree": degrees,
        "description_length": description_lengths,
        "inaccuracy": inaccuracies,
        "surfeit": surfeits,
    }
)

The usual error of a polynomial model tends to decrease as the degree grows. Surfeit adds a complementary view: it measures whether the description is becoming unnecessarily long or redundant relative to the target representation.


## 4. External serialization: decision tree versus neural network

These notebook helpers demonstrate explicit model-description strings. Use `surfeit_model()` or `model_analysis()` to evaluate fitted models through canonical library serialization.


In [ ]:
def decision_tree_to_string(model, feature_names=None) -> str:
    """Serialize a fitted decision tree using scikit-learn's text exporter."""
    if feature_names is None:
        feature_names = [f"x{i}" for i in range(model.n_features_in_)]
    return export_text(model, feature_names=feature_names)


def mlp_to_string(model) -> str:
    """Serialize a fitted MLP by recording architecture, weights, and biases."""
    lines = ["MLPClassifier"]
    lines.append(f"hidden_layer_sizes={model.hidden_layer_sizes}")
    lines.append(f"activation={model.activation}")
    lines.append(f"solver={model.solver}")

    for layer_index, weights in enumerate(model.coefs_):
        lines.append(f"weights_layer_{layer_index}={np.array2string(weights, precision=4, separator=',')}")

    for layer_index, biases in enumerate(model.intercepts_):
        lines.append(f"biases_layer_{layer_index}={np.array2string(biases, precision=4, separator=',')}")

    return "\n".join(lines)

In [ ]:
X, y = make_blobs(
    n_samples=600,
    centers=2,
    n_features=2,
    cluster_std=1.0,
    random_state=42,
)

plt.scatter(X[:, 0], X[:, 1], c=y, s=10)
plt.xlabel("x0")
plt.ylabel("x1")
plt.title("Simple binary classification problem")
plt.show()

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X, y)

nn = MLPClassifier(
    hidden_layer_sizes=(20,),
    max_iter=1000,
    random_state=42,
)
nn.fit(X, y)

print("Decision tree training accuracy:", tree.score(X, y))
print("Neural network training accuracy:", nn.score(X, y))

In [ ]:
sft = Surfeit(y_type="categorical")
sft.fit_y(y)

tree_description = decision_tree_to_string(tree)
nn_description = mlp_to_string(nn)

comparison = pd.DataFrame(
    [
        {
            "model": "Decision tree",
            "description_length": len(tree_description.encode("utf-8")),
            "surfeit": sft.surfeit_string(tree_description),
        },
        {
            "model": "Neural network",
            "description_length": len(nn_description.encode("utf-8")),
            "surfeit": sft.surfeit_string(nn_description),
        },
    ]
)

comparison


Both models can perform well on this simple problem, but their descriptions can have very different lengths and redundancy. The exact result depends on the external serialization function, which is why serialization is now kept outside the metric class.


## 5. Same predictions, different descriptions

Surfeit can also be used to compare two descriptions that implement the same rule. This is useful when evaluating alternative model representations, generated code, or explanatory descriptions.


In [ ]:
minimal_description = """def classify(x):
    return int(x[0] > 0)
"""

verbose_description = """def classify(x):
    # Initialize several variables that are not really needed.
    first_coordinate = x[0]
    threshold = 0
    comparison = first_coordinate > threshold

    # Convert the Boolean comparison into an integer class.
    if comparison is True:
        predicted_class = 1
    else:
        predicted_class = 0

    # Store and return the final answer.
    result = predicted_class
    return result
"""

sft = Surfeit(y_type="categorical")
sft.fit_y(y)

pd.DataFrame(
    [
        {
            "description": "Minimal",
            "length": len(minimal_description.encode("utf-8")),
            "surfeit": sft.surfeit_string(minimal_description),
        },
        {
            "description": "Verbose",
            "length": len(verbose_description.encode("utf-8")),
            "surfeit": sft.surfeit_string(verbose_description),
        },
    ]
)


## 6. Edge case: constant target

If the target contains no empirical information, then any non-empty model description is surplus relative to the target. This is an important sanity check.


In [ ]:
constant_y = np.ones(100, dtype=int)
constant_model = """def model(x):
    return 1
"""

Surfeit(y_type="categorical").fit_y(constant_y).surfeit_string(constant_model)


A surfeit value of 1 means that the model description is entirely unnecessary relative to a constant target representation.


## Summary

`Surfeit` evaluates descriptive complexity through canonical fitted-model descriptions or explicit strings:

- fit the target representation,
- serialize a fitted model or accept a description string,
- compress that string,
- return a scalar surfeit value or an analysis dictionary.

`pformat(report)` produces a formatted numerical dictionary with code lengths in bits and the reference used in the calculation. The dictionary retains the complete diagnostics.
